In [3]:
pip install pyarrow

  Using cached pyarrow-23.0.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
    --------------------------------------- 0.5/27.5 MB 2.1 MB/s eta 0:00:13
   - -------------------------------------- 1.3/27.5 MB 3.0 MB/s eta 0:00:09
   --- ------------------------------------ 2.1/27.5 MB 3.4 MB/s eta 0:00:08
   ---- ----------------------------------- 2.9/27.5 MB 3.4 MB/s eta 0:00:08
   ---- ----------------------------------- 3.1/27.5 MB 3.3 MB/s eta 0:00:08
   ----- ---------------------------------- 3.9/27.5 MB 3.1 MB/s eta 0:00:08
   ------ --------------------------------- 4.5/27.5 MB 3.0 MB/s eta 0:00:08
   ------- -------------------------------- 5.5/27.5 MB 3.3 MB/s eta 0:00:07
   --------- ------------------------------ 6.3/27.5 MB 3.4 MB/s eta 0:00:07
   ---------- ---------

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

# Настройки путей (под твою структуру)
RAW_DATA_PATH = '../data/raw/application_train.csv'
PROCESSED_DATA_DIR = '../data/processed'
PROCESSED_FILE_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_cleaned.csv')

def reduce_mem_usage(df):
    """
    Итерируется по всем колонкам и меняет тип данных на самый экономичный
    в зависимости от диапазона значений.
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Исходный размер памяти: {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object and col_type.name != 'category':
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                # Для float используем float32, так как float16 не всегда поддерживается моделями
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            # Объекты переводим в категорию для экономии памяти во время работы
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Конечный размер памяти: {end_mem:.2f} MB')
    print(f'Уменьшено на: {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

def clean_and_process(df):
    # 1. Аномалии стажа (365243 -> NaN)
    df['DAYS_EMPLOYED_ANOM'] = (df["DAYS_EMPLOYED"] == 365243).astype(np.int8)
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    
    # 2. Удаляем пустые колонки (где заполнено менее 40% данных = пропущено более 60%)
    # В твоем примере LIMIT - это порог "оставить", если не-NA значений >= limit
    limit = len(df) * 0.4 
    df = df.dropna(thresh=limit, axis=1)
    
    # 3. Фильтруем пол (удаляем 4 строчки с XNA)
    df = df[df['CODE_GENDER'] != 'XNA']
    
    # 4. Базовая предобработка категорий для CSV
    # Чтобы потом удобно читать CSV, лучше сразу сделать One-Hot Encoding (get_dummies)
    # ИЛИ (для экономии места в CSV) оставить как есть, а кодировать при обучении.
    # Для диплома и простоты CSV я рекомендую оставить как есть, но XGBoost требует чисел.
    # Сделаем Factorize (Label Encoding) для объектов, чтобы в CSV были числа, а не длинные строки.
    
    obj_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in obj_cols:
        # Кодируем строки в числа, пропуски остаются -1 (или NaN)
        df[col], _ = pd.factorize(df[col])
        
    return df

if __name__ == "__main__":
    print("Загрузка данных...")
    # engine='c' быстрее
    df = pd.read_csv(RAW_DATA_PATH, engine='c')
    
    print("Обработка данных...")
    df = clean_and_process(df)
    
    print("Оптимизация памяти...")
    df = reduce_mem_usage(df)
    
    # Сохранение
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
    
    print(f"Сохранение в {PROCESSED_FILE_PATH}...")
    # index=False важно, чтобы не создавать лишнюю колонку Unnamed: 0
    df.to_csv(PROCESSED_FILE_PATH, index=False)
    
    print("Готово!")
    # Сборка мусора
    del df
    gc.collect()

Загрузка данных...
Обработка данных...
Оптимизация памяти...
Исходный размер памяти: 248.98 MB
Конечный размер памяти: 78.89 MB
Уменьшено на: 68.3%
Сохранение в ../data/processed\train_cleaned.csv...
Готово!
